# CS5494 Final Project — Restaurant Review Rating

本 Notebook 按照实验流程逐步完成三条评分路线：

| Task | 方法 | 说明 |
|------|------|------|
| **A** | Zero-shot | 直接用基础模型推理 |
| **B.1.1** | N-shot ICL | 给模型提供少量示例后推理 |
| **B.1.2** | LoRA 微调 | 指令微调后推理 |

**运行前请确认**：
- 已执行 `pip install -r requirements.txt`
- `data/` 目录下包含 `review_train.csv`、`review_test.csv`、`test_answer.csv`
- （可选）如需从 ModelScope 下载模型，请设置 `USE_MODELSCOPE = True`

---
## 0. 全局配置

所有超参数、路径、开关统一在此 cell 设置，后续 cell 均使用这里的变量。

In [ ]:
import sys, os
# notebook 在 notebooks/ 子目录，把项目根目录加入 Python 路径
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

MODEL_NAME      = "Qwen/Qwen2.5-0.5B-Instruct"
# MODEL_NAME      = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_NAME      = "Qwen/Qwen2.5-3B-Instruct"
USE_MODELSCOPE  = True                            # True = ModelScope 镜像下载
USE_QLORA       = True                           # [QLoRA] True=4-bit量化训练
MODEL_CACHE_DIR = "../models"

DATA_DIR        = "../data"
SEED            = 42

DEBUG_TRAIN_SIZE = None
DEBUG_TEST_SIZE  = None

N_SHOT          = 4        # few-shot 示例数
N_PER_RATING    = 10       # 每个评分从训练集中收集多少个示例

LORA_CHECKPOINT_DIR = "../qwen_lora_checkpoints"
LORA_FINAL_DIR      = "../qwen_lora_final"

MAX_NEW_TOKENS  = 10       # 推理时最多生成 token 数

OVERSAMPLE = True          # 开启过采样
BATCHSIZE = 8

model_short_name  = MODEL_NAME.split('/')[-1]
OUTPUT_DIR      = f"../outputs/QLoRa/v2/{model_short_name}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"配置完成 ✓ (结果将保存在 {OUTPUT_DIR})")


---
## 1. 数据加载

`load_project_data` 会自动读取三份 CSV 并将文本评分（如 `"4 star"`）转换为数值列 `rating_numeric`（1-5）。

In [ ]:
from dataset import load_project_data

train_df, test_df, answer_df = load_project_data(DATA_DIR)

print(f"训练集: {len(train_df)} 条")
print(f"测试集: {len(test_df)} 条")
print(f"答案集: {len(answer_df)} 条")
train_df.head(3)

### 1.1 标签分布

查看训练集各评分类别的样本数，确认数据是否均衡。

In [ ]:
print("训练集评分分布:")
print(train_df["rating_numeric"].value_counts().sort_index())
print()
print("答案集评分分布:")
print(answer_df["rating_numeric"].value_counts().sort_index())

---
## 2. 划分训练集 / 验证集

使用分层切分（stratify）保证各评分类别在训练集和验证集中比例一致。

- `test_size=0.2`：20% 作为验证集
- `debug_sample_size`：设置后会对切分结果进行随机截断，用于快速调试

In [ ]:
from dataset import split_train_val

train_split, val_split = split_train_val(
    train_df,
    seed=SEED,
    test_size=0.2,
    debug_sample_size=DEBUG_TRAIN_SIZE,
)

print(f"训练子集: {len(train_split)} 条")
print(f"验证子集: {len(val_split)} 条")

### 调试用：截断测试集

推理很慢时，可以先在小样本上验证流程。

In [ ]:
if DEBUG_TEST_SIZE is not None:
    test_df  = test_df.sample(n=min(DEBUG_TEST_SIZE, len(test_df)), random_state=SEED).reset_index(drop=True)
    answer_df = answer_df[answer_df["Review_id"].isin(test_df["Review_id"])].reset_index(drop=True)
    print(f"调试模式：测试集截断为 {len(test_df)} 条")
else:
    print(f"全量测试集：{len(test_df)} 条")

---
## 3. 加载基础模型

`load_tokenizer_and_model` 会自动检测设备（CUDA / NPU / MPS / CPU）并选择合适的 dtype。

返回值：
- `tokenizer`：分词器
- `base_model`：未微调的基础模型
- `model_path`：实际加载路径
- `device`：`DeviceInfo` 对象，含设备类型和显存信息

In [ ]:
from model import load_tokenizer_and_model

tokenizer, base_model, model_path, device = load_tokenizer_and_model(
    model_name=MODEL_NAME,
    use_modelscope=USE_MODELSCOPE,
    cache_dir=MODEL_CACHE_DIR,
    use_qlora=USE_QLORA,  # [QLoRA]
)

print(f"设备类型 : {device.device_type}")
print(f"设备名称 : {device.device_name}")
if device.total_memory_gb:
    print(f"显存容量 : {device.total_memory_gb:.1f} GB")
print(f"模型路径 : {model_path}")

---
## 4. Task A — Zero-shot 推理

直接使用基础模型，不提供任何示例。

**Prompt 格式**：描述评分标准 + 评论文本，要求模型输出单个数字（1-5）。

返回的 `zero_out` 是一个字典：
- `predictions_df`：含 `Review_id` 和 `Predicted_Rating` 的 DataFrame
- `debug_records`：每条样本的原始输出和解析详情
- `inference_seconds`：总推理耗时

In [ ]:
import time
import pandas as pd
from model import build_zero_shot_prompt, generate_response, extract_rating_from_output

sample_row = test_df.iloc[6]
print(f'Review_id : {sample_row["Review_id"]}')
print(f'Title     : {sample_row.get("Title")}')
print(f'Review    : {str(sample_row["Review"])}')
true_answer = answer_df[answer_df['Review_id'] == sample_row['Review_id']]['rating_numeric'].values[0]
print(f'True_ans  : {true_answer}')


In [ ]:
# 第 1 步：构建 prompt
sample_title = '' if pd.isna(sample_row.get('Title')) else str(sample_row.get('Title'))
sample_review = str(sample_row['Review'])
sample_prompt = build_zero_shot_prompt(title=sample_title, review=sample_review)
print(f'Prompt: \n'+sample_prompt)


In [ ]:
# 第 2 步：调用模型生成
sample_raw_output = generate_response(
    model=base_model,
    tokenizer=tokenizer,
    prompt=sample_prompt,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f'模型原始输出: '+ repr(sample_raw_output))

In [ ]:
# 第 3 步：解析评分
sample_predicted = extract_rating_from_output(sample_raw_output)
print(f'解析结果: {sample_predicted}')

In [ ]:
ids = []
predictions = []
debug_records = []

start_time = time.time()

for step, (_, row) in enumerate(test_df.iterrows()):
    review_id = row['Review_id']
    title = '' if pd.isna(row.get('Title')) else str(row.get('Title'))
    review = str(row['Review'])

    # 第 1 步：构建 prompt
    prompt = build_zero_shot_prompt(title=title, review=review)

    # 第 2 步：调用模型生成
    raw_output = generate_response(
        model=base_model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    # 第 3 步：从模型输出中解析评分（1-5），解析失败默认为 3
    predicted_rating = extract_rating_from_output(raw_output)

    # 收集结果
    ids.append(review_id)
    predictions.append(predicted_rating)
    debug_records.append({
        'sample_idx': step,
        'review_id': review_id,
        'prompt': prompt,
        'raw_output': raw_output,
        'predicted_rating': predicted_rating,
    })

    # 每 10 条打印一次进度
    if (step + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f'已处理 {step + 1}/{len(test_df)} 条，耗时 {elapsed:.1f}s')

total_time = time.time() - start_time

# 汇总结果
zero_predictions_df = pd.DataFrame({'Review_id': ids, 'Predicted_Rating': predictions})

zero_out = {
    'predictions_df': zero_predictions_df,
    'debug_records': debug_records,
    'inference_seconds': total_time,
    'sample_count': len(predictions),
}

print(f'\n推理完成：{zero_out["sample_count"]} 条')
print(f'总耗时  ：{zero_out["inference_seconds"]:.1f}s')
print(f'平均耗时：{zero_out["inference_seconds"] / zero_out["sample_count"]:.2f}s/样本')
zero_out['predictions_df'].head()

### 4.1 评估 Zero-shot 结果

In [ ]:
from eval import evaluate_predictions

zero_metric = evaluate_predictions(zero_out["predictions_df"], answer_df)

print(f"Accuracy    : {zero_metric['accuracy']:.4f}")
print(f"Macro-F1    : {zero_metric['macro_f1']:.4f}")
print(f"Weighted-F1 : {zero_metric['weighted_f1']:.4f}")
print()
print(zero_metric["classification_report"])

### 4.2 混淆矩阵

In [ ]:
import pandas as pd

cm_zero = pd.DataFrame(
    zero_metric["confusion_matrix"],
    index=[f"True {i}★" for i in range(1, 6)],
    columns=[f"Pred {i}★" for i in range(1, 6)],
)
print("Zero-shot 混淆矩阵:")
cm_zero

### 4.3 保存预测结果

In [ ]:
zero_out["predictions_df"].to_csv(f"{OUTPUT_DIR}/zero_shot_predictions.csv", index=False)
print("已保存 zero_shot_predictions.csv")

---
## 5. Task B.1.1 — N-shot In-Context Learning

在同一基础模型上，在 prompt 中插入 N 个带标注的示例（few-shot examples），引导模型仿照示例格式输出评分。

**示例库构建**：`build_example_library` 从训练集中每个评分等级各采样 `n_per_rating` 条，共最多 5×N 条。

推理时每条测试样本会从示例库中随机选 `n_shot` 条插入 prompt，使用确定性 hash seed 保证可复现。

In [ ]:
import pandas as pd
import hashlib
from dataset import build_example_library
from model import build_nshot_prompt, generate_response, extract_rating_from_output

# 构建示例库
example_lib = build_example_library(train_split, n_per_rating=N_PER_RATING, seed=SEED)
print(f"示例库大小：{len(example_lib)} 条（每评分最多 {N_PER_RATING} 条）")

In [ ]:
# 取测试集第 1 行
sample_row = test_df.iloc[6]
review_id = sample_row["Review_id"]
print(f'Review_id : {review_id}')
print(f'Title     : {sample_row.get("Title")}')
print(f'Review    : {str(sample_row["Review"])}')
true_answer = answer_df[answer_df['Review_id'] == sample_row['Review_id']]['rating_numeric'].values[0]
print(f'True_ans  : {true_answer}')

In [ ]:
# 第 1 步：哈希计算种子 & 构建 prompt
review_id_text = str(review_id)
stable_hash = int(hashlib.md5(review_id_text.encode("utf-8")).hexdigest()[:8], 16)
row_seed = SEED + (stable_hash % 1_000_000)

sample_title = '' if pd.isna(sample_row.get('Title')) else str(sample_row.get('Title'))
sample_review = str(sample_row['Review'])
sample_prompt = build_nshot_prompt(
    title=sample_title, 
    review=sample_review, 
    examples=example_lib,
    n=N_SHOT,
    seed=row_seed
)
print(f'动态种子: {row_seed}')
print(f'N-shot Prompt: \n' + sample_prompt)

In [ ]:
# 第 2 步：调用模型生成
sample_raw_output = generate_response(
    model=base_model,
    tokenizer=tokenizer,
    prompt=sample_prompt,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f'模型原始输出: '+repr(sample_raw_output))

In [ ]:
# 第 3 步：解析评分
sample_predicted = extract_rating_from_output(sample_raw_output)
print(f'解析结果: {sample_predicted}')

In [ ]:
ids = []
predictions = []
debug_records = []

start_time = time.time()

for step, (_, row) in enumerate(test_df.iterrows()):
    review_id = row['Review_id']
    title = '' if pd.isna(row.get('Title')) else str(row.get('Title'))
    review = str(row['Review'])

    review_id_text = str(review_id)
    stable_hash = int(hashlib.md5(review_id_text.encode("utf-8")).hexdigest()[:8], 16)
    row_seed = SEED + (stable_hash % 1_000_000)

    # 第 1 步：构建 prompt
    prompt = build_nshot_prompt(
        title=title, 
        review=review, 
        examples=example_lib,
        n=N_SHOT,
        seed=row_seed
    )

    # 第 2 步：调用模型生成
    raw_output = generate_response(
        model=base_model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    # 第 3 步：从模型输出中解析评分（1-5），解析失败默认为 3
    predicted_rating = extract_rating_from_output(raw_output)

    # 收集结果
    ids.append(review_id)
    predictions.append(predicted_rating)
    debug_records.append({
        'sample_idx': step,
        'review_id': review_id,
        'prompt': prompt,
        'raw_output': raw_output,
        'predicted_rating': predicted_rating,
    })

    # 每 10 条打印一次进度
    if (step + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f'已处理 {step + 1}/{len(test_df)} 条，耗时 {elapsed:.1f}s')

total_time = time.time() - start_time

# 汇总结果
nshot_predictions_df = pd.DataFrame({'Review_id': ids, 'Predicted_Rating': predictions})

nshot_out = {
    'predictions_df': nshot_predictions_df,
    'debug_records': debug_records,
    'inference_seconds': total_time,
    'sample_count': len(predictions),
}

print(f'\n推理完成：{nshot_out["sample_count"]} 条')
print(f'总耗时  ：{nshot_out["inference_seconds"]:.1f}s')
print(f'平均耗时：{nshot_out["inference_seconds"] / nshot_out["sample_count"]:.2f}s/样本')


### 5.1 评估 N-shot 结果

In [ ]:
nshot_metric = evaluate_predictions(nshot_out["predictions_df"], answer_df)

print(f"Accuracy    : {nshot_metric['accuracy']:.4f}")
print(f"Macro-F1    : {nshot_metric['macro_f1']:.4f}")
print(f"Weighted-F1 : {nshot_metric['weighted_f1']:.4f}")
print()
print(nshot_metric["classification_report"])

### 5.2 混淆矩阵

In [ ]:
cm_nshot = pd.DataFrame(
    nshot_metric["confusion_matrix"],
    index=[f"True {i}★" for i in range(1, 6)],
    columns=[f"Pred {i}★" for i in range(1, 6)],
)
print(f"{N_SHOT}-shot 混淆矩阵:")
cm_nshot

### 5.3 保存预测结果

In [ ]:
nshot_out["predictions_df"].to_csv(f"{OUTPUT_DIR}/nshot_predictions.csv", index=False)
print(f"已保存 nshot_predictions.csv")

---
## 6. Task B.1.2 — LoRA 微调

对基础模型进行指令微调（SFT），使用 PEFT 的 LoRA 方法只训练少量参数。

**训练数据格式**（`prepare_instruction_data` 的输出）：
```
{ "instruction": "...", "input": "Title: ...\nReview: ...", "output": "4" }
```

**关键超参数**（可在 `LoraTrainingConfig` 中修改）：

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `lora_rank` | 4 | LoRA 低秩矩阵的秩 |
| `lora_alpha` | 8 | LoRA 缩放系数 |
| `num_train_epochs` | 3 | 训练轮数 |
| `train_batch_size` | 8 | 每个设备 batch 大小 |
| `learning_rate` | 5e-5 | 学习率 |

In [ ]:
from dataset import prepare_instruction_data
from train import LoraTrainingConfig, train_lora_model

# 构建指令样本
train_records = prepare_instruction_data(train_split, oversample = OVERSAMPLE)
val_records   = prepare_instruction_data(val_split)
print(f"训练指令样本: {len(train_records)} 条")
print(f"验证指令样本: {len(val_records)} 条")

# 示例：打印第一条样本
print("\n第一条训练样本:")
for k, v in train_records[0].items():
    print(f"{k}: {v[:80]}{'...' if len(v) > 80 else ''}")

### 6.1 配置 & 启动训练

> ⚠️ **注意**：训练会覆盖 `base_model` 的 adapter 状态。若需重跑 Task A/B.1.1，请重新执行 Section 3 重新加载基础模型。

In [ ]:
cfg = LoraTrainingConfig(
    output_dir=LORA_CHECKPOINT_DIR,
    final_dir=LORA_FINAL_DIR,
    use_qlora=USE_QLORA,  # [QLoRA]
    seed = SEED,
    # lora_rank=16,
    # 调试时可减少训练样本和 epoch
    # max_train_samples=100,
    # num_train_epochs=1,
)

train_res = train_lora_model(
    base_model=base_model,
    tokenizer=tokenizer,
    train_records=train_records,
    val_records=val_records,
    config=cfg,
)

print(f"训练完成 ✓")
print(f"总耗时      : {train_res['train_seconds']:.1f}s")
print(f"训练样本数  : {train_res['train_samples']}")
print(f"验证样本数  : {train_res['val_samples']}")
print(f"模型保存路径: {train_res['output_dir']}")

---
## 7. Task B.1.2 — LoRA 推理

将 LoRA adapter 合并（merge）到基础模型权重中，再进行推理。

**Merge 的好处**：推理速度与原始模型相同，无 adapter 额外开销。

> 若已有训练好的 adapter，可以直接修改 `lora_dir` 跳过 Section 6 直接执行本节。

In [ ]:
from model import load_merged_lora_model

lora_dir = train_res["output_dir"]

tok_lora, merged_model, lora_device = load_merged_lora_model(
    base_model_name=MODEL_NAME,
    lora_dir=lora_dir,
    use_modelscope=USE_MODELSCOPE,
    cache_dir=MODEL_CACHE_DIR,
)

print(f"LoRA merge 完成 ✓  设备: {lora_device.device_type}")

## LoRA微调后推理

In [ ]:
import pandas as pd
from model import build_finetuned_prompt, generate_response, extract_rating_from_output

# 取测试集
sample_row = test_df.iloc[6]
review_id = sample_row["Review_id"]
print(f'Review_id : {review_id}')
print(f'Title     : {sample_row.get("Title")}')
print(f'Review    : {str(sample_row["Review"])[:200]}...')
true_answer = answer_df[answer_df['Review_id'] == sample_row['Review_id']]['rating_numeric'].values[0]
print(f'True_ans  : {true_answer}')

In [ ]:
# 第 1 步：构建微调模型专用的 Prompt
sample_title = '' if pd.isna(sample_row.get('Title')) else str(sample_row.get('Title'))
sample_review = str(sample_row['Review'])
sample_prompt = build_finetuned_prompt(title=sample_title, review=sample_review)
print(f'Zero-shot Prompt:\n' + sample_prompt)

In [ ]:
# 调用微调后的模型生成（挂载 LoRA 权重）
sample_raw_output = generate_response(
    model=merged_model,
    tokenizer=tok_lora,
    prompt=sample_prompt,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f'模型原始输出: ' + repr(sample_raw_output))

In [ ]:
# 第 3 步：解析评级数字
sample_predicted = extract_rating_from_output(sample_raw_output)
print(f'Rating:  {sample_predicted}')

In [ ]:
import time
import pandas as pd
from model import build_finetuned_prompt, generate_response, extract_rating_from_output

ids = []
predictions = []
debug_records = []

start_time = time.time()

for step, (_, row) in enumerate(test_df.iterrows()):
    review_id = row['Review_id']
    title = '' if pd.isna(row.get('Title')) else str(row.get('Title'))
    review = str(row['Review'])

    # 第 1 步：构建 Prompt (Zero-shot)
    prompt = build_finetuned_prompt(title=title, review=review)

    # 第 2 步：调用模型生成
    raw_output = generate_response(
        model=merged_model,
        tokenizer=tok_lora,       
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    # 第 3 步：解析评分
    predicted_rating = extract_rating_from_output(raw_output)

    # 收集结果
    ids.append(review_id)
    predictions.append(predicted_rating)
    debug_records.append({
        'sample_idx': step,
        'review_id': review_id,
        'prompt': prompt,
        'raw_output': raw_output,
        'predicted_rating': predicted_rating,
    })

    # 每 10 条打印一次进度
    if (step + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f'已处理 {step + 1}/{len(test_df)} 条，耗时 {elapsed:.1f}s')

total_time = time.time() - start_time

# 汇总结果打包
lora_predictions_df = pd.DataFrame({'Review_id': ids, 'Predicted_Rating': predictions})
lora_out = run_lora_inference_batch(
    lora_model=merged_model,
    tokenizer=tok_lora,
    test_df=test_df,
    batch_size= BATCHSIZE,
    max_new_tokens=10,
)

print(f'\n推理完成：{lora_out["sample_count"]} 条')
print(f'总耗时  ：{lora_out["inference_seconds"]:.1f}s')
print(f'平均耗时：{lora_out["inference_seconds"] / lora_out["sample_count"]:.2f}s/样本')

### 7.1 评估 LoRA 结果

In [ ]:
from eval import evaluate_predictions
lora_metric = evaluate_predictions(lora_out["predictions_df"], answer_df)

print(f"Accuracy    : {lora_metric['accuracy']:.4f}")
print(f"Macro-F1    : {lora_metric['macro_f1']:.4f}")
print(f"Weighted-F1 : {lora_metric['weighted_f1']:.4f}")
print()
print(lora_metric["classification_report"])

### 7.2 混淆矩阵

In [ ]:
cm_lora = pd.DataFrame(
    lora_metric["confusion_matrix"],
    index=[f"True {i}★" for i in range(1, 6)],
    columns=[f"Pred {i}★" for i in range(1, 6)],
)
print("LoRA 混淆矩阵:")
cm_lora

display(lora_metric["confusion_matrix_fig"])

### 7.3 保存预测结果

In [ ]:
lora_out["predictions_df"].to_csv(f"{OUTPUT_DIR}/lora_predictions.csv", index=False)
print("已保存 lora_predictions.csv")

---
## 8. 结果汇总对比

将三条路线的指标汇总到一张表，方便撰写报告。

In [ ]:
comparison = pd.DataFrame([
    {
        "Method"     : "Task A: Zero-shot",
        "Accuracy"   : round(zero_metric["accuracy"],   4),
        "Macro-F1"   : round(zero_metric["macro_f1"],   4),
        "Weighted-F1": round(zero_metric["weighted_f1"],4),
        "Time(s)"    : round(zero_out["inference_seconds"], 1),
        "s/sample"   : round(zero_out["inference_seconds"] / zero_out["sample_count"], 3),
    },
    {
        "Method"     : f"Task B.1.1: {N_SHOT}-shot ICL",
        "Accuracy"   : round(nshot_metric["accuracy"],   4),
        "Macro-F1"   : round(nshot_metric["macro_f1"],   4),
        "Weighted-F1": round(nshot_metric["weighted_f1"],4),
        "Time(s)"    : round(nshot_out["inference_seconds"], 1),
        "s/sample"   : round(nshot_out["inference_seconds"] / nshot_out["sample_count"], 3),
    },
    {
        "Method"     : "Task B.1.2: LoRA Fine-tuned",
        "Accuracy"   : round(lora_metric["accuracy"],   4),
        "Macro-F1"   : round(lora_metric["macro_f1"],   4),
        "Weighted-F1": round(lora_metric["weighted_f1"],4),
        "Time(s)"    : round(lora_out["inference_seconds"], 1),
        "s/sample"   : round(lora_out["inference_seconds"] / lora_out["sample_count"], 3),
    },
])

comparison

In [ ]:
comparison.to_csv(f"{OUTPUT_DIR}/comparison_results.csv", index=False)
print("已保存 comparison_results.csv")

---
## 9. Debug：查看原始模型输出

对比不同方法对同一条评论的原始输出，便于分析模型行为。

In [ ]:
# 查看详细预测 debug 信息
print("=== Zero-shot debug ===")
for rec in zero_out["debug_records"]:
    print(f"[{rec['review_id']}] pred={rec['predicted_rating']}  raw='{rec['raw_output']}'")

print()
print("=== N-shot debug ===")
for rec in nshot_out["debug_records"]:
    print(f"[{rec['review_id']}] pred={rec['predicted_rating']}  raw='{rec['raw_output']}'")

print()
print("=== LoRA debug ===")
for rec in lora_out["debug_records"]:
    print(f"[{rec['review_id']}] pred={rec['predicted_rating']}  raw='{rec['raw_output']}'")